In [32]:
%%configure -f
{
    "executorMemory": "1g",
    "executorCores": 1,
    "driverMemory": "1g",
    "driverCores": 1,
    "numExecutors": 1
}

StatementMeta(sparkpool, 58, -1, Finished, Available, Finished, False)

See https://go.microsoft.com/fwlink/?linkid=2170827


In [33]:
data = spark.read.format("delta").load("abfss://silver@adlstoragesiddharth.dfs.core.windows.net/fraud_detection/clean_transactions/")

display(data)

StatementMeta(sparkpool, 58, 2, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77f1a17c-39b4-4d9b-8cd5-ac2d0201dbb3)

In [34]:
data.printSchema()

StatementMeta(sparkpool, 58, 3, Finished, Available, Finished, False)

root
 |-- transaction_id: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- transaction_location: string (nullable = true)
 |-- customer_home_location: string (nullable = true)
 |-- distance_from_home: double (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- account_balance: double (nullable = true)
 |-- daily_transaction_count: double (nullable = true)
 |-- weekly_transaction_count: double (nullable = true)
 |-- avg_transaction_amount: double (nullable = true)
 |-- max_transaction_last_24h: double (nullable = true)
 |-- is_international_transaction: string (nullable = true)
 |-- is_ne

In [35]:
dim_customer = spark.sql("""
                SELECT 
                customer_id,
                customer_home_location,
                MIN(transaction_date) AS valid_from,
                NULL AS valid_to
                FROM {df}
                GROUP BY customer_id, customer_home_location
""", df=data)

#display(dim_customer)

StatementMeta(sparkpool, 58, 4, Finished, Available, Finished, False)

In [36]:
dim_customer.write.format("delta").mode("overwrite").save("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_customer")

StatementMeta(sparkpool, 58, 5, Finished, Available, Finished, False)

In [37]:
dim_merchant = spark.sql("""
                    SELECT 
                    merchant_id,
                    merchant_category,
                    MIN(transaction_date) AS valid_from,
                    NULL AS valid_to
                    FROM {df}
                    GROUP BY merchant_id, merchant_category
""", df=data)

#display(dim_merchant)

# Save to gold
dim_merchant.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_merchant")


StatementMeta(sparkpool, 58, 6, Finished, Available, Finished, False)

In [38]:
dim_device = spark.sql("""
                SELECT 
                device_id,
                MIN(transaction_date) AS valid_from,
                NULL AS valid_to
  FROM {df}
  GROUP BY device_id
""", df=data)

#display(dim_device)

# Save to gold
dim_device.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_device")


StatementMeta(sparkpool, 58, 7, Finished, Available, Finished, False)

In [39]:
dim_card = spark.sql(""" SELECT
            ROW_NUMBER() OVER(PARTITION BY CUSTOMER_ID ORDER BY CARD_TYPE) AS CARD_ID,
            CUSTOMER_ID,
            CARD_TYPE,
            MIN(TRANSACTION_DATE) AS VALID_FROM,
            NULL AS VALID_TO
            FROM {df}
            GROUP BY CUSTOMER_ID, CARD_TYPE
""",
df=data)

#display(dim_card)
# Save to gold
dim_card.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_card")


StatementMeta(sparkpool, 58, 8, Finished, Available, Finished, False)

In [41]:
dim_location = spark.sql(""" SELECT
                    ROW_NUMBER() OVER(ORDER BY TRANSACTION_LOCATION) AS LOCATION_ID,
                    TRANSACTION_LOCATION,
                    MIN(TRANSACTION_DATE) AS VALID_FROM
                    FROM {df}
                    WHERE TRANSACTION_LOCATION IS NOT NULL
                    GROUP BY TRANSACTION_LOCATION

""", df=data) 
#display(dim_location)

# Save to gold
dim_location.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_location")


StatementMeta(sparkpool, 58, 10, Finished, Available, Finished, False)

In [43]:
dim_date = spark.sql("""
  SELECT 
    date_val AS date_id,
    date_format(date_val, 'EEEE') AS day_of_week,
    CASE WHEN dayofweek(date_val) IN (1, 7) THEN 'Yes' ELSE 'No' END AS is_weekend,
    'No' AS is_holiday,
    month(date_val) AS month,
    quarter(date_val) AS quarter
  FROM (
    SELECT explode(sequence(to_date('2025-01-01'), to_date('2025-05-01'))) AS date_val
  )
  ORDER BY date_val
""")

#display(dim_date)

# Save
dim_date.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_date"
)


StatementMeta(sparkpool, 58, 12, Finished, Available, Finished, False)

In [44]:
## FACT TABLES

StatementMeta(sparkpool, 58, 13, Finished, Available, Finished, False)

In [45]:
from pyspark.sql.functions import current_timestamp

# Read ALL dimensions from gold
dim_customer = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_customer")
dim_merchant = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_merchant")
dim_device = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_device")
dim_card = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_card")
dim_location = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_location")
dim_date = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_date")

# Create fact_transactions with ALL joins
fact_transactions = (data
  .join(dim_customer, data.customer_id == dim_customer.customer_id, "left")
  .join(dim_merchant, data.merchant_id == dim_merchant.merchant_id, "left")
  .join(dim_device, data.device_id == dim_device.device_id, "left")
  .join(dim_card, (data.customer_id == dim_card.CUSTOMER_ID) & (data.card_type == dim_card.CARD_TYPE), "left")
  .join(dim_location, data.transaction_location == dim_location.TRANSACTION_LOCATION, "left")
  .join(dim_date, data.transaction_date == dim_date.date_id, "left")
  .select(
    data.transaction_id,
    data.customer_id,
    data.merchant_id,
    dim_card.CARD_ID,
    dim_device.device_id,
    dim_location.LOCATION_ID,
    dim_date.date_id,
    data.transaction_amount,
    data.transaction_type,
    data.is_international_transaction,
    data.fraud_label,
    data.unusual_time_transaction,
    data.failed_transaction_count,
    data.distance_from_home,
    data.previous_fraud_count,
    data.daily_transaction_count,
    data.weekly_transaction_count,
    data.max_transaction_last_24h,
    data.account_balance,
    current_timestamp().alias("load_timestamp")
  )
)

display(fact_transactions)

# Save
fact_transactions.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_transactions"
)



StatementMeta(sparkpool, 58, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 870c6b2e-30a4-49ef-be9b-0a9f42418c7d)

In [46]:
fact_customer_daily_summary = spark.sql("""
    SELECT 
        customer_id,
        transaction_date,
        COUNT(transaction_id) AS transaction_count,
        SUM(transaction_amount) AS total_amount,
        SUM(CASE WHEN fraud_label = 'Fraud' THEN 1 ELSE 0 END) AS fraud_count,
        AVG(transaction_amount) AS avg_amount
    FROM {df}
    GROUP BY customer_id, transaction_date
""", df=data)

display(fact_customer_daily_summary)

fact_customer_daily_summary.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_customer_daily_summary"
)



StatementMeta(sparkpool, 58, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d294afd1-9a51-4e1d-9d67-52fc61d15211)

In [47]:
# FACT TABLE 3: fact_merchant_daily_summary
fact_merchant_daily_summary = spark.sql("""
    SELECT 
        merchant_id,
        merchant_category,
        transaction_date,
        COUNT(transaction_id) AS transaction_count,
        SUM(transaction_amount) AS total_amount,
        SUM(CASE WHEN fraud_label = 'Fraud' THEN 1 ELSE 0 END) AS fraud_count,
        AVG(transaction_amount) AS avg_amount,
        current_timestamp() AS load_timestamp
    FROM {df}
    GROUP BY merchant_id, merchant_category, transaction_date
""", df=data)

display(fact_merchant_daily_summary)

fact_merchant_daily_summary.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_merchant_daily_summary"
)


StatementMeta(sparkpool, 58, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10f9b76e-27b4-4a58-8203-47a81ea92060)

In [48]:
fact_device_suspicious_activity = spark.sql("""
    SELECT 
        device_id,
        transaction_date,
        COUNT(transaction_id) AS transaction_count,
        SUM(CASE WHEN fraud_label = 'Fraud' THEN 1 ELSE 0 END) AS fraud_count,
        COUNT(DISTINCT transaction_location) AS unique_locations,
        SUM(CASE WHEN failed_transaction_count > 0 THEN 1 ELSE 0 END) AS failed_attempts,
        current_timestamp() AS load_timestamp
    FROM {df}
    GROUP BY device_id, transaction_date
""", df=data)

display(fact_device_suspicious_activity)

fact_device_suspicious_activity.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_device_suspicious_activity"
)


StatementMeta(sparkpool, 58, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0eb50a62-c9ad-4903-921e-d5e30a4cfeb4)

In [49]:
fact_anomalies_detected = spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY transaction_id) AS anomaly_id,
        transaction_id,
        customer_id,
        transaction_date AS detected_date,
        CASE WHEN fraud_label = 'Fraud' THEN 'fraud' ELSE 'suspicious' END AS anomaly_type,
        CASE WHEN fraud_label = 'Fraud' THEN 'High' ELSE 'Medium' END AS severity,
        current_timestamp() AS detected_timestamp,
        'No' AS alert_sent
    FROM {df}
    WHERE fraud_label = 'Fraud' 
       OR unusual_time_transaction = 'Yes' 
       OR distance_from_home > 500
""", df=data)

display(fact_anomalies_detected)

fact_anomalies_detected.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_anomalies_detected"
)


StatementMeta(sparkpool, 58, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b861abf3-7d14-4650-9596-bc1eef03e306)

In [50]:
fact_fraud_cases = spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY transaction_id) AS fraud_case_id,
        transaction_id,
        customer_id,
        merchant_id,
        transaction_date,
        'Open' AS case_status,
        NULL AS resolved_date,
        'Under Investigation' AS resolution_type,
        current_timestamp() AS created_date
    FROM {df}
    WHERE fraud_label = 'Fraud'
""", df=data)

display(fact_fraud_cases)

fact_fraud_cases.write.format("delta").mode("overwrite").save(
    "abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_fraud_cases"
)


StatementMeta(sparkpool, 58, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6b5e46b-fd45-453a-a327-324c9d511f4d)

In [51]:
# JDBC Connection to Azure SQL Database
jdbc_url = "jdbc:sqlserver://sqldbc37.database.windows.net:1433;database=fraud_transaction_dataset;user=siddharthdb;password=Sid@1805;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

# Read dimensions from gold layer
dim_customer = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_customer")
dim_merchant = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_merchant")
dim_device = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_device")
dim_card = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_card")
dim_location = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_location")
dim_date = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/dimensions/dim_date")

# Read facts from gold layer
fact_transactions = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_transactions")
fact_customer_daily_summary = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_customer_daily_summary")
fact_merchant_daily_summary = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_merchant_daily_summary")
fact_device_suspicious_activity = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_device_suspicious_activity")
fact_anomalies_detected = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_anomalies_detected")
fact_fraud_cases = spark.read.format("delta").load("abfss://gold@adlstoragesiddharth.dfs.core.windows.net/Financial_Transaction_Fraud_Dataset/facts/fact_fraud_cases")



StatementMeta(sparkpool, 58, 20, Finished, Available, Finished, False)

In [52]:
sql_server = mssparkutils.credentials.getSecret("project1db", "sqldb")
sql_username = mssparkutils.credentials.getSecret("project1db", "sqlusername")
sql_password = mssparkutils.credentials.getSecret("project1db", "azsqldb")

StatementMeta(sparkpool, 58, 21, Finished, Available, Finished, False)

In [53]:
jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database=fraud_transaction_dataset;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"


dim_customer.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_customer")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()

print("dim_customer saved")

StatementMeta(sparkpool, 58, 22, Finished, Available, Finished, False)

dim_customer saved


In [54]:
# DIMENSIONS
dim_customer.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_customer")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" dim_customer saved")

dim_merchant.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_merchant")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print("dim_merchant saved")

dim_device.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_device")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" dim_device saved")

dim_card.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_card")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" dim_card saved")

dim_location.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_location")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print("dim_location saved")

dim_date.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.dim_date")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" dim_date saved")

# FACTS
fact_transactions.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_transactions")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print("fact_transactions saved")

fact_customer_daily_summary.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_customer_daily_summary")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" fact_customer_daily_summary saved")

fact_merchant_daily_summary.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_merchant_daily_summary")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" fact_merchant_daily_summary saved")

fact_device_suspicious_activity.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_device_suspicious_activity")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" fact_device_suspicious_activity saved")

fact_anomalies_detected.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_anomalies_detected")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" fact_anomalies_detected saved")

fact_fraud_cases.write.format("jdbc")\
    .option("url", jdbc_url)\
    .option("dbtable", "dbo.fact_fraud_cases")\
    .option("user", sql_username)\
    .option("password", sql_password)\
    .mode("ignore").save()
print(" fact_fraud_cases saved")

print("\n ALL 12 TABLES WRITTEN TO SQL DATABASE!")

StatementMeta(sparkpool, 58, 23, Finished, Available, Finished, False)

 dim_customer saved
dim_merchant saved
 dim_device saved
 dim_card saved
dim_location saved
 dim_date saved
fact_transactions saved
 fact_customer_daily_summary saved
 fact_merchant_daily_summary saved
 fact_device_suspicious_activity saved
 fact_anomalies_detected saved
 fact_fraud_cases saved

 ALL 12 TABLES WRITTEN TO SQL DATABASE!
